In [93]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, OrdinalEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import plotly.graph_objects as go
import html
import uuid
data = pd.read_csv('data_visualization.csv')

The goal for this file is to repeat the creation of my dashboard while taking into account all the changes we discussed listed in ProjectIteration_Diary.txt. Note I relied on LLMS to make my plots interactable and styled.

In [94]:
data = data.dropna(subset=["ticker", "publisher"])
data["publisher"] = data["publisher"].astype(str).str.strip()
data["ticker"] = data["ticker"].astype(str).str.strip()

numeric_cols = [
    "sentiment_encoded",
    "return_30d",
    "return_6m",
    "long_term",
    "accuracy"
]

for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["accuracy"] = data["accuracy"].fillna(0)
data["sentiment_encoded"] = data["sentiment_encoded"].fillna(0)
data["long_term"] = data["long_term"].fillna(0)

In [95]:
display(HTML("""
<style>
    .dashboard-shell {
        background: #0b0f14;
        color: #f4f7fb;
        padding: 22px;
        border-radius: 18px;
        border: 1px solid #1f2937;
        font-family: Inter, Arial, sans-serif;
        box-shadow: 0 8px 24px rgba(0,0,0,0.25);
        margin-top: 10px;
    }

    .dashboard-title {
        font-size: 30px;
        font-weight: 700;
        letter-spacing: 0.4px;
        margin-bottom: 4px;
        color: #f9fafb;
    }

    .dashboard-subtitle {
        font-size: 14px;
        color: #9ca3af;
        margin-bottom: 18px;
    }

    .section-title {
        font-size: 18px;
        font-weight: 700;
        color: #f9fafb;
        margin-top: 18px;
        margin-bottom: 12px;
        letter-spacing: 0.2px;
    }

    .metric-grid {
        display: grid;
        grid-template-columns: repeat(4, minmax(180px, 1fr));
        gap: 12px;
        margin-bottom: 16px;
    }

    .metric-card {
        background: linear-gradient(180deg, #111827 0%, #0f172a 100%);
        border: 1px solid #1f2937;
        border-radius: 14px;
        padding: 14px 16px;
        box-shadow: 0 6px 18px rgba(0,0,0,0.28);
    }

    .metric-label {
        font-size: 12px;
        color: #9ca3af;
        margin-bottom: 8px;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }

    .metric-value {
        font-size: 22px;
        font-weight: 700;
        color: #f9fafb;
    }

    .metric-note {
        font-size: 12px;
        color: #34d399;
        margin-top: 6px;
    }

    .summary-box {
        background: linear-gradient(180deg, #111827 0%, #0f172a 100%);
        border: 1px solid #1f2937;
        border-radius: 14px;
        padding: 16px;
        margin-bottom: 16px;
    }

    .summary-title {
        font-size: 16px;
        font-weight: 700;
        color: #f9fafb;
        margin-bottom: 8px;
    }

    .summary-text {
        font-size: 14px;
        color: #d1d5db;
        line-height: 1.6;
    }

    table.dark-sortable {
        width: 100%;
        border-collapse: collapse;
        font-size: 13px;
        margin-top: 8px;
        background: #0f172a;
        color: #f3f4f6;
        border-radius: 12px;
        overflow: hidden;
    }

    table.dark-sortable th,
    table.dark-sortable td {
        border-bottom: 1px solid #1f2937;
        padding: 10px 12px;
        text-align: left;
    }

    table.dark-sortable th {
        background: #111827;
        color: #e5e7eb;
        cursor: pointer;
        position: sticky;
        top: 0;
        z-index: 1;
        user-select: none;
    }

    table.dark-sortable tr:hover td {
        background: #111827;
    }

    .table-wrap {
        max-height: 420px;
        overflow-y: auto;
        border: 1px solid #1f2937;
        border-radius: 12px;
        margin-bottom: 16px;
    }

    .small-note {
        font-size: 12px;
        color: #9ca3af;
        margin-top: 6px;
        margin-bottom: 12px;
    }

    /* ---- ipywidgets dark styling ---- */
    .dark-controls {
        background: #111827;
        border: 1px solid #1f2937;
        border-radius: 14px;
        padding: 14px;
        margin: 10px 0 16px 0;
    }

    .dark-widget label,
    .dark-widget .widget-label {
        color: #e5e7eb !important;
        font-weight: 600 !important;
        min-width: fit-content !important;
    }

    .dark-widget select,
    .dark-widget input[type="range"],
    .dark-widget .widget-readout {
        background: #0f172a !important;
        color: #f9fafb !important;
        border-color: #334155 !important;
    }

    .dark-widget .widget-dropdown > select {
        background: #0f172a !important;
        color: #f9fafb !important;
        border: 1px solid #334155 !important;
        border-radius: 8px !important;
    }

    .dark-widget .widget-intslider .noUi-target,
    .dark-widget .widget-intslider input {
        background: #0f172a !important;
    }

    .dark-widget .widget-readout {
        border-radius: 6px !important;
        padding: 2px 6px !important;
    }

    /* helps the output area blend with the dark dashboard */
    .jp-OutputArea-output,
    .jupyter-widgets-output-area {
        background: transparent !important;
    }
</style>
"""))

In [96]:
ticker_features = data.groupby("ticker").agg(
    avg_sentiment=("sentiment_encoded", "mean"),
    sentiment_std=("sentiment_encoded", "std"),
    bullish_rate=("sentiment_encoded", lambda x: (x == 1).mean()),
    bearish_rate=("sentiment_encoded", lambda x: (x == -1).mean()),
    neutral_rate=("sentiment_encoded", lambda x: (x == 0).mean()),
    avg_return_30d=("return_30d", "mean"),
    avg_return_6m=("return_6m", "mean"),
    vol_30d=("return_30d", "std"),
    article_count=("headline", "count"),
    long_term_ratio=("long_term", "mean")
).reset_index()

ticker_features["sentiment_std"] = ticker_features["sentiment_std"].fillna(0)
ticker_features["vol_30d"] = ticker_features["vol_30d"].fillna(0)
ticker_features["log_article_count"] = np.log1p(ticker_features["article_count"])

In [97]:
publisher_overall = data.groupby("publisher").agg(
    overall_articles=("headline", "count"),
    overall_accuracy=("accuracy", "mean"),
    overall_avg_sentiment=("sentiment_encoded", "mean"),
    overall_long_term_ratio=("long_term", "mean"),
    unique_tickers=("ticker", "nunique")
).reset_index()

publisher_overall["log_overall_articles"] = np.log1p(publisher_overall["overall_articles"])

publisher_by_ticker = data.groupby(["ticker", "publisher"]).agg(
    ticker_articles=("headline", "count"),
    ticker_accuracy=("accuracy", "mean"),
    ticker_avg_sentiment=("sentiment_encoded", "mean"),
    ticker_long_term_ratio=("long_term", "mean")
).reset_index()

publisher_lookup = publisher_by_ticker.merge(
    publisher_overall,
    on="publisher",
    how="left"
)


In [98]:
publisher_cluster_cols = [
    "overall_accuracy",
    "overall_avg_sentiment",
    "log_overall_articles",
    "unique_tickers",
    "overall_long_term_ratio"
]

publisher_X = publisher_overall[publisher_cluster_cols].fillna(0)

publisher_scaler = StandardScaler()
publisher_X_scaled = publisher_scaler.fit_transform(publisher_X)

publisher_kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
publisher_overall["cluster"] = publisher_kmeans.fit_predict(publisher_X_scaled)

publisher_silhouette = silhouette_score(publisher_X_scaled, publisher_overall["cluster"])

publisher_pca = PCA(n_components=2, random_state=42)
publisher_pca_vals = publisher_pca.fit_transform(publisher_X_scaled)

publisher_overall["pca1"] = publisher_pca_vals[:, 0]
publisher_overall["pca2"] = publisher_pca_vals[:, 1]

publisher_lookup = publisher_lookup.merge(
    publisher_overall[["publisher", "cluster", "pca1", "pca2"]],
    on="publisher",
    how="left"
)

cluster_summary = publisher_overall.groupby("cluster").agg(
    overall_accuracy=("overall_accuracy", "mean"),
    overall_articles=("overall_articles", "mean"),
    overall_avg_sentiment=("overall_avg_sentiment", "mean"),
    unique_tickers=("unique_tickers", "mean")
).reset_index()

accuracy_median = publisher_overall["overall_accuracy"].median()
articles_median = publisher_overall["overall_articles"].median()
sentiment_median = publisher_overall["overall_avg_sentiment"].median()
coverage_median = publisher_overall["unique_tickers"].median()

cluster_descriptions = {}

for _, row in cluster_summary.iterrows():
    cluster_id = int(row["cluster"])

    parts = []

    if row["overall_accuracy"] >= accuracy_median:
        parts.append("higher accuracy")
    else:
        parts.append("lower accuracy")

    if row["overall_articles"] >= articles_median:
        parts.append("high volume")
    else:
        parts.append("lower volume")

    if row["unique_tickers"] >= coverage_median:
        parts.append("broad coverage")
    else:
        parts.append("more specialized coverage")

    if row["overall_avg_sentiment"] >= sentiment_median:
        parts.append("more positive sentiment")
    else:
        parts.append("more neutral/negative sentiment")

    cluster_descriptions[cluster_id] = ", ".join(parts).capitalize() + "."



In [99]:
ticker_dropdown = widgets.Dropdown(
    options=sorted(ticker_features["ticker"].unique()),
    value=sorted(ticker_features["ticker"].unique())[0],
    description="Ticker:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

publisher_dropdown = widgets.Dropdown(
    options=["All Publishers"] + sorted(publisher_overall["publisher"].unique()),
    value="All Publishers",
    description="Publisher:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px")
)

min_articles_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=max(1, int(publisher_lookup["ticker_articles"].max())),
    step=1,
    description="Min Articles on Ticker:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="380px")
)

controls = widgets.HBox(
    [ticker_dropdown, publisher_dropdown, min_articles_slider],
    layout=widgets.Layout(gap="14px", flex_flow="row wrap")
)

dashboard_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #1f2937",
        padding="16px",
        margin="0",
        width="100%"
    )
)
dashboard_output.add_class("dark-controls")


Output(layout=Layout(border_bottom='1px solid #1f2937', border_left='1px solid #1f2937', border_right='1px sol…

In [100]:
def fmt_pct(x):
    return f"{x:.1%}" if pd.notnull(x) else ""

def fmt_num(x, digits=3):
    return f"{x:.{digits}f}" if pd.notnull(x) else ""

def make_metric_card(label, value, note=""):
    return f"""
    <div class="metric-card">
        <div class="metric-label">{html.escape(label)}</div>
        <div class="metric-value">{html.escape(str(value))}</div>
        <div class="metric-note">{html.escape(note)}</div>
    </div>
    """

def dataframe_to_sortable_html(df, title=None):
    table_id = f"tbl_{uuid.uuid4().hex[:10]}"
    df_html = df.to_html(
        index=False,
        classes="dark-sortable",
        table_id=table_id,
        escape=False
    )

    js = f"""
    <script>
    (function() {{
        const table = document.getElementById("{table_id}");
        if (!table) return;

        const getCellValue = (tr, idx) => {{
            const text = tr.children[idx].innerText.trim().replace(/,/g, "");
            const num = parseFloat(text.replace("%",""));
            return isNaN(num) ? text.toLowerCase() : num;
        }};

        const comparer = function(idx, asc) {{
            return function(a, b) {{
                const v1 = getCellValue(asc ? a : b, idx);
                const v2 = getCellValue(asc ? b : a, idx);
                return v1 > v2 ? 1 : v1 < v2 ? -1 : 0;
            }}
        }};

        Array.from(table.querySelectorAll("th")).forEach(function(th, idx) {{
            let asc = true;
            th.addEventListener("click", function() {{
                const tbody = table.querySelector("tbody");
                Array.from(tbody.querySelectorAll("tr"))
                    .sort(comparer(idx, asc = !asc))
                    .forEach(tr => tbody.appendChild(tr));
            }});
        }});
    }})();
    </script>
    """

    title_html = ""
    if title is not None:
        title_html = f'<div class="section-title">{html.escape(title)}</div>'

    return f"""
    {title_html}
    <div class="table-wrap">
        {df_html}
    </div>
    {js}
    """

def build_publisher_scatter(selected_ticker, selected_publisher, min_articles):
    plot_df = publisher_lookup[publisher_lookup["ticker"] == selected_ticker].copy()
    plot_df = plot_df[plot_df["ticker_articles"] >= min_articles].copy()

    if plot_df.empty:
        fig = go.FigureWidget()
        fig.update_layout(
            template="plotly_dark",
            height=460,
            title="Publisher Accuracy vs Sentiment for Selected Ticker",
            paper_bgcolor="#0b0f14",
            plot_bgcolor="#111827",
            annotations=[
                dict(
                    text="No publishers match the current filter.",
                    x=0.5, y=0.5,
                    xref="paper", yref="paper",
                    showarrow=False,
                    font=dict(size=16)
                )
            ]
        )
        return fig

    plot_df = plot_df.sort_values("ticker_articles", ascending=False)

    sizes = np.interp(
        plot_df["ticker_articles"],
        (plot_df["ticker_articles"].min(), plot_df["ticker_articles"].max()),
        (14, 40)
    ) if plot_df["ticker_articles"].nunique() > 1 else np.full(len(plot_df), 24)

    fig = go.FigureWidget()

    fig.add_scatter(
        x=plot_df["ticker_avg_sentiment"],
        y=plot_df["ticker_accuracy"],
        mode="markers",
        marker=dict(
            size=sizes,
            color=plot_df["ticker_articles"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Articles"),
            line=dict(width=1, color="#d1d5db"),
            opacity=0.92
        ),
        text=plot_df["publisher"],
        customdata=np.stack([
            plot_df["publisher"],
            plot_df["ticker_articles"],
            plot_df["overall_articles"],
            plot_df["overall_accuracy"],
            plot_df["ticker_accuracy"]
        ], axis=1),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Avg Sentiment on Ticker: %{x:.3f}<br>"
            "Ticker Accuracy: %{y:.3f}<br>"
            "Articles on Ticker: %{customdata[1]}<br>"
            "Overall Articles: %{customdata[2]}<br>"
            "Overall Accuracy: %{customdata[3]:.3f}<extra></extra>"
        ),
        name="Publishers"
    )

    if selected_publisher != "All Publishers":
        selected_row = plot_df[plot_df["publisher"] == selected_publisher]
        if not selected_row.empty:
            fig.add_scatter(
                x=selected_row["ticker_avg_sentiment"],
                y=selected_row["ticker_accuracy"],
                mode="markers+text",
                text=selected_row["publisher"],
                textposition="top center",
                marker=dict(
                    size=28,
                    symbol="diamond",
                    color="#f59e0b",
                    line=dict(width=2, color="white")
                ),
                hoverinfo="skip",
                name="Selected Publisher"
            )

    fig.update_layout(
        template="plotly_dark",
        height=500,
        title=f"Selected Ticker: {selected_ticker} | Publisher Accuracy vs Sentiment",
        paper_bgcolor="#0b0f14",
        plot_bgcolor="#111827",
        font=dict(color="#f9fafb"),
        xaxis=dict(
            title="Average Sentiment on Ticker",
            gridcolor="#253047",
            zerolinecolor="#334155"
        ),
        yaxis=dict(
            title="Ticker Accuracy",
            gridcolor="#253047",
            zerolinecolor="#334155",
            range=[-0.02, 1.02]
        ),
        margin=dict(l=50, r=40, t=60, b=50),
        legend=dict(orientation="h", y=1.08, x=0.01)
    )

    scatter_trace = fig.data[0]

    def handle_click(trace, points, selector):
        if points.point_inds:
            idx = points.point_inds[0]
            clicked_publisher = plot_df.iloc[idx]["publisher"]
            publisher_dropdown.value = clicked_publisher

    scatter_trace.on_click(handle_click)

    return fig


def build_publisher_cluster_plot(selected_ticker, selected_publisher):
    plot_df = publisher_overall.copy()

    highlight_publishers = set(
        publisher_lookup[publisher_lookup["ticker"] == selected_ticker]["publisher"].unique()
    )

    plot_df["in_selected_ticker"] = plot_df["publisher"].isin(highlight_publishers)

    fig = go.FigureWidget()

    fig.add_scatter(
        x=plot_df["pca1"],
        y=plot_df["pca2"],
        mode="markers",
        marker=dict(
            size=np.interp(
                plot_df["overall_articles"],
                (plot_df["overall_articles"].min(), plot_df["overall_articles"].max()),
                (10, 32)
            ) if plot_df["overall_articles"].nunique() > 1 else np.full(len(plot_df), 18),
            color=plot_df["cluster"],
            colorscale="Turbo",
            showscale=True,
            colorbar=dict(title="Cluster"),
            line=dict(
                width=np.where(plot_df["in_selected_ticker"], 2.5, 0.8),
                color=np.where(plot_df["in_selected_ticker"], "#f8fafc", "#475569")
            ),
            opacity=0.9
        ),
        customdata=np.stack([
            plot_df["publisher"],
            plot_df["overall_accuracy"],
            plot_df["overall_articles"],
            plot_df["overall_avg_sentiment"],
            plot_df["cluster"]
        ], axis=1),
        text=plot_df["publisher"],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Cluster: %{customdata[4]}<br>"
            "Overall Accuracy: %{customdata[1]:.3f}<br>"
            "Overall Articles: %{customdata[2]}<br>"
            "Overall Avg Sentiment: %{customdata[3]:.3f}<extra></extra>"
        ),
        name="Publishers"
    )

    if selected_publisher != "All Publishers":
        selected_row = plot_df[plot_df["publisher"] == selected_publisher]
        if not selected_row.empty:
            fig.add_scatter(
                x=selected_row["pca1"],
                y=selected_row["pca2"],
                mode="markers+text",
                text=selected_row["publisher"],
                textposition="top center",
                marker=dict(
                    size=30,
                    symbol="star",
                    color="#22c55e",
                    line=dict(width=2, color="white")
                ),
                hoverinfo="skip",
                name="Selected Publisher"
            )

    fig.update_layout(
        template="plotly_dark",
        height=520,
        title=f"Publisher PCA / Clustering  |  Silhouette Score = {publisher_silhouette:.3f}",
        paper_bgcolor="#0b0f14",
        plot_bgcolor="#111827",
        font=dict(color="#f9fafb"),
        xaxis=dict(title="PCA Component 1", gridcolor="#253047"),
        yaxis=dict(title="PCA Component 2", gridcolor="#253047"),
        margin=dict(l=50, r=40, t=60, b=50),
        legend=dict(orientation="h", y=1.08, x=0.01)
    )

    scatter_trace = fig.data[0]

    def handle_click(trace, points, selector):
        if points.point_inds:
            idx = points.point_inds[0]
            clicked_publisher = plot_df.iloc[idx]["publisher"]
            publisher_dropdown.value = clicked_publisher

    scatter_trace.on_click(handle_click)

    return fig


def build_accuracy_table(selected_ticker, min_articles):
    table_df = publisher_lookup[publisher_lookup["ticker"] == selected_ticker].copy()
    table_df = table_df[table_df["ticker_articles"] >= min_articles].copy()

    if table_df.empty:
        return pd.DataFrame({"Message": ["No publisher rows available for this filter."]})

    table_df = table_df[[
        "publisher",
        "ticker_articles",
        "overall_articles",
        "overall_accuracy",
        "ticker_accuracy"
    ]].copy()

    table_df = table_df.rename(columns={
        "publisher": "Publisher",
        "ticker_articles": "Articles on Ticker",
        "overall_articles": "Overall Articles",
        "overall_accuracy": "Overall Accuracy",
        "ticker_accuracy": "Ticker Accuracy"
    })

    table_df["Overall Accuracy"] = table_df["Overall Accuracy"].map(lambda x: f"{x:.3f}")
    table_df["Ticker Accuracy"] = table_df["Ticker Accuracy"].map(lambda x: f"{x:.3f}")

    return table_df.sort_values("Articles on Ticker", ascending=False)


def build_global_stats_table():
    table_df = publisher_overall[[
        "publisher",
        "cluster",
        "overall_articles",
        "unique_tickers",
        "overall_accuracy",
        "overall_avg_sentiment",
        "overall_long_term_ratio"
    ]].copy()

    table_df = table_df.rename(columns={
        "publisher": "Publisher",
        "cluster": "Cluster",
        "overall_articles": "Overall Articles",
        "unique_tickers": "Unique Tickers",
        "overall_accuracy": "Overall Accuracy",
        "overall_avg_sentiment": "Overall Avg Sentiment",
        "overall_long_term_ratio": "Long-Term Ratio"
    })

    table_df["Overall Accuracy"] = table_df["Overall Accuracy"].map(lambda x: f"{x:.3f}")
    table_df["Overall Avg Sentiment"] = table_df["Overall Avg Sentiment"].map(lambda x: f"{x:.3f}")
    table_df["Long-Term Ratio"] = table_df["Long-Term Ratio"].map(lambda x: f"{x:.3f}")

    return table_df.sort_values("Overall Accuracy", ascending=False)


def build_summary_html(selected_ticker, selected_publisher, min_articles):
    ticker_row = ticker_features[ticker_features["ticker"] == selected_ticker].iloc[0]

    ticker_publishers = publisher_lookup[
        (publisher_lookup["ticker"] == selected_ticker) &
        (publisher_lookup["ticker_articles"] >= min_articles)
    ].copy()

    n_publishers = len(ticker_publishers)
    top_accuracy = ticker_publishers["ticker_accuracy"].max() if not ticker_publishers.empty else np.nan
    avg_accuracy = ticker_publishers["ticker_accuracy"].mean() if not ticker_publishers.empty else np.nan

    cards_html = f"""
    <div class="metric-grid">
        {make_metric_card("Selected Ticker", selected_ticker, "Current dashboard focus")}
        {make_metric_card("Ticker Articles", int(ticker_row["article_count"]), "All articles in dataset")}
        {make_metric_card("Avg Sentiment", f"{ticker_row['avg_sentiment']:.3f}", "Mean sentiment for selected ticker")}
        {make_metric_card("Avg 30d Return", f"{ticker_row['avg_return_30d']:.3f}", "Mean 30-day return after articles")}
        {make_metric_card("Avg 6m Return", f"{ticker_row['avg_return_6m']:.3f}", "Mean 6-month return after articles")}
        {make_metric_card("30d Volatility", f"{ticker_row['vol_30d']:.3f}", "Std. dev. of 30-day return")}
        {make_metric_card("Publishers Shown", int(n_publishers), f"Filtered at ≥ {min_articles} article(s)")}
        {make_metric_card("Best Ticker Accuracy", f"{top_accuracy:.3f}" if pd.notnull(top_accuracy) else "N/A", "Top publisher for this ticker")}
    </div>
    """

    publisher_summary_html = ""
    if selected_publisher != "All Publishers":
        overall_row = publisher_overall[publisher_overall["publisher"] == selected_publisher]
        ticker_row_pub = publisher_lookup[
            (publisher_lookup["ticker"] == selected_ticker) &
            (publisher_lookup["publisher"] == selected_publisher)
        ]

        if not overall_row.empty:
            overall_row = overall_row.iloc[0]
            cluster_id = int(overall_row["cluster"])
            cluster_text = cluster_descriptions.get(cluster_id, "No description available.")

            ticker_specific_text = ""
            if not ticker_row_pub.empty:
                ticker_row_pub = ticker_row_pub.iloc[0]
                ticker_specific_text = (
                    f"For <b>{html.escape(selected_ticker)}</b>, this publisher has "
                    f"<b>{int(ticker_row_pub['ticker_articles'])}</b> article(s), "
                    f"<b>{ticker_row_pub['ticker_accuracy']:.3f}</b> ticker accuracy, and "
                    f"<b>{ticker_row_pub['ticker_avg_sentiment']:.3f}</b> average ticker sentiment."
                )
            else:
                ticker_specific_text = (
                    f"This publisher does not have rows for <b>{html.escape(selected_ticker)}</b> "
                    f"under the current filter."
                )

            publisher_summary_html = f"""
            <div class="summary-box">
                <div class="summary-title">Selected Publisher Summary</div>
                <div class="summary-text">
                    <b>{html.escape(selected_publisher)}</b> belongs to publisher cluster <b>{cluster_id}</b>.<br><br>
                    Cluster profile: {html.escape(cluster_text)}<br><br>
                    Overall accuracy: <b>{overall_row['overall_accuracy']:.3f}</b><br>
                    Overall average sentiment: <b>{overall_row['overall_avg_sentiment']:.3f}</b><br>
                    Overall articles: <b>{int(overall_row['overall_articles'])}</b><br>
                    Unique tickers covered: <b>{int(overall_row['unique_tickers'])}</b><br><br>
                    {ticker_specific_text}
                </div>
            </div>
            """

    return cards_html + publisher_summary_html

In [101]:
def update_dashboard(change=None):
    selected_ticker = ticker_dropdown.value
    selected_publisher = publisher_dropdown.value
    min_articles = min_articles_slider.value

    with dashboard_output:
        clear_output(wait=True)

        display(HTML('<div class="section-title">Ticker / Publisher Summary</div>'))
        display(HTML(build_summary_html(selected_ticker, selected_publisher, min_articles)))

        display(HTML('<div class="section-title">Average Sentiment vs Ticker Accuracy by Publisher</div>'))
        display(HTML('<div class="small-note">Each point is a publisher covering the selected ticker. Bubble size/color reflects article count on that ticker.</div>'))
        display(build_publisher_scatter(selected_ticker, selected_publisher, min_articles))

        display(HTML('<div class="section-title">PCA / Clustering of Publishers</div>'))
        display(HTML('<div class="small-note">Publishers are grouped using overall accuracy, sentiment, article volume, ticker breadth, and long-term ratio.</div>'))
        display(build_publisher_cluster_plot(selected_ticker, selected_publisher))

        accuracy_table = build_accuracy_table(selected_ticker, min_articles)
        display(HTML(
            dataframe_to_sortable_html(
                accuracy_table,
                title=f"Publisher Coverage + Accuracy for {selected_ticker}"
            )
        ))

        global_stats_table = build_global_stats_table()
        display(HTML(
            dataframe_to_sortable_html(
                global_stats_table,
                title="Global Publisher Sentiment"
            )
        ))

In [102]:
ticker_dropdown.observe(update_dashboard, names="value")
publisher_dropdown.observe(update_dashboard, names="value")
min_articles_slider.observe(update_dashboard, names="value")
ticker_dropdown.add_class("dark-widget")
publisher_dropdown.add_class("dark-widget")
min_articles_slider.add_class("dark-widget")
controls.add_class("dark-controls")

header_html = widgets.HTML("""
<div class="dashboard-shell">
    <div class="dashboard-title">Financial News Source Dashboard</div>
    <div class="dashboard-subtitle">
        Click a point in either plot to update the Publisher dropdown.
        Use the Ticker dropdown to switch the dashboard context.
    </div>
</div>
""")

app = widgets.VBox(
    [header_html, controls, dashboard_output],
    layout=widgets.Layout(width="100%")
)

display(app)
update_dashboard()

In [103]:
# saving my basic clustering model 

import joblib

deployment_bundle = {
    "publisher_cluster_cols": publisher_cluster_cols,
    "scaler": publisher_scaler,
    "kmeans": publisher_kmeans,
    "pca": publisher_pca,
    "silhouette_score": publisher_silhouette,
    "n_clusters": publisher_kmeans.n_clusters
}

joblib.dump(deployment_bundle, "publisher_clustering_model.joblib")

print("Model bundle saved as publisher_clustering_model.joblib")

Model bundle saved as publisher_clustering_model.joblib
